# EdgePCB-QC — Hailo-8 & Hailo-8L HEF Conversion Pipeline
**Bharat AI-SoC Challenge 2026–27 | PS5: Real-Time Edge AI PCB Defect Inspection**  
**Hailo Dataflow Compiler (DFC) 3.34.0 Production Compilation Notebook**

---

### Notebook Purpose & Conversion Flow
This turnkey Google Colab notebook compiles the frozen **EdgePCB-QC YOLOv8n** vision baseline (`pcb_multidomain_yolov8n_640.onnx`) into a **Hailo Executable Format (`.hef`)** binary for deployment on the **Raspberry Pi AI HAT+**:
1. **Target Hardware:** Hailo-8 (26 TOPS, 8-lane) or Hailo-8L (13 TOPS, 4-lane PCIe)
2. **Input Model:** Ultralytics YOLOv8n (OpSet 12, static `[1, 3, 640, 640]`, 6 defect classes)
3. **Pipeline Stages:**
   $$\text{ONNX (FP32)} \xrightarrow{\text{Parse}} \text{HAR} \xrightarrow{\text{Quantize (INT8)}} \text{Quantized HAR} \xrightarrow{\text{Compile}} \text{HEF}$$
4. **Quantization Scheme:** Post-Training Quantization (PTQ) to INT8 with 256–512 authentic PCB calibration frames.
5. **Verified Class Taxonomy (6 classes):**
   - `0: open_circuit` (Critical)
   - `1: short_circuit` (Critical)
   - `2: mouse_bite` (Major)
   - `3: spur` (Minor)
   - `4: spurious_copper` (Major)
   - `5: pin_hole_missing_pad` (Minor)

> **Important Safety Notice:**  
> This notebook performs parsing, calibration, optimization, and compilation. Real NPU inference requires physical Hailo-8 hardware (Raspberry Pi AI HAT+). Do not claim HEF runtime functionality until benchmarked on hardware with `hailortcli`.


## 1. System Environment & Python Verification
Hailo Dataflow Compiler 3.34.0 is compiled as a C++/Python extension for **x86_64 Linux** and **Python 3.10** (or 3.8/3.9).  
Google Colab typically defaults to Python 3.10 or 3.11. This cell audits the system architecture, OS release, and Python version.


In [ ]:
import sys
import platform
import os
import subprocess

print("=" * 70)
print("SYSTEM & PYTHON AUDIT FOR HAILO DFC 3.34.0")
print("=" * 70)

arch = platform.machine()
os_info = platform.platform()
py_ver = sys.version.split()[0]
py_major_minor = f"{sys.version_info.major}.{sys.version_info.minor}"

print(f"[*] Architecture    : {arch} (Required: x86_64)")
print(f"[*] Operating System: {os_info}")
print(f"[*] Python Version  : {py_ver}")

if arch != "x86_64":
    raise SystemError(f"[CRITICAL] Hailo DFC requires an x86_64 host! Detected: {arch}")

print("[PASS] Host architecture verified: x86_64.")

# Check for GPU (DFC quantization uses GPU if available for faster profiling)
has_gpu = subprocess.run(["which", "nvidia-smi"], stdout=subprocess.PIPE).returncode == 0
if has_gpu:
    print("[*] NVIDIA GPU Detected:")
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
else:
    print("[NOTE] Running on CPU. DFC will use multi-threaded CPU for quantization (sufficient).")


## 2. Mount Google Drive & Establish Workspace Directory
All input files (ONNX model, calibration dataset, DFC wheel) are loaded from Google Drive, and compiled artifacts (`.har`, `.hef`, logs) are written directly to persistent Drive storage.


In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# Define project directories
GDRIVE_DIR = Path('/content/drive/MyDrive/EdgePCB-QC_Hailo')
LOCAL_WORK_DIR = Path('/content/edgepcb_hailo_workspace')

GDRIVE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"[*] Persistent Google Drive Directory : {GDRIVE_DIR}")
print(f"[*] High-Speed Local Workspace        : {LOCAL_WORK_DIR}")

# List existing assets in Google Drive folder
print("
[*] Files currently in Google Drive folder:")
for item in GDRIVE_DIR.glob('*'):
    size_mb = item.stat().st_size / (1024 * 1024)
    print(f"    - {item.name:<45} ({size_mb:.2f} MB)")


## 3. Install Hailo Dataflow Compiler (DFC 3.34.0)
Place the official `hailo_dataflow_compiler-3.34.0-py3-none-linux_x86_64.whl` (or your available version from Hailo Developer Zone) into your Google Drive folder:  
`/content/drive/MyDrive/EdgePCB-QC_Hailo/`

This cell automatically locates the wheel, installs necessary system and Python dependencies (`graphviz`, `pydot`, `netron`, `onnx`), and installs the DFC.


In [ ]:
import sys
import glob
from pathlib import Path

# Search for DFC wheel in Google Drive or local workspace
wheels = list(GDRIVE_DIR.glob("*hailo_dataflow_compiler*.whl")) + list(Path('/content').glob("*hailo_dataflow_compiler*.whl"))

if not wheels:
    print("[WARNING] No hailo_dataflow_compiler-*.whl found in Google Drive folder!")
    print(f"Please upload your 'hailo_dataflow_compiler-3.34.0-py3-none-linux_x86_64.whl' to:")
    print(f"  {GDRIVE_DIR}")
    print("
If you are using a direct download link, you can download it via:")
    print("  !wget -O /content/hailo_dataflow_compiler.whl '<YOUR_HAILO_DOWNLOAD_URL>'")
else:
    dfc_wheel = sorted(wheels)[-1]
    print(f"[+] Found Hailo DFC wheel: {dfc_wheel.name}")
    
    # 1. Install prerequisites
    print("
[*] Installing system and python prerequisites...")
    !apt-get update -qq && apt-get install -y -qq graphviz libgraphviz-dev
    !pip install -q graphviz pydot onnx onnxruntime opencv-python-headless tabulate
    
    # 2. Install DFC wheel
    print(f"
[*] Installing {dfc_wheel.name}...")
    !pip install -q "{dfc_wheel}"
    
    print("
[+] Hailo DFC installation completed successfully.")


## 4. Verify Hailo DFC Installation & CLI Tools
Audits the installed `hailo` CLI tool, Python SDK import, and supported target architectures (`hailo8`, `hailo8l`).


In [ ]:
# Verify CLI tool
!hailo --version

# Verify Python SDK client
try:
    import hailo_sdk_client
    print(f"[PASS] hailo_sdk_client loaded successfully. Version: {hailo_sdk_client.__version__}")
    
    from hailo_sdk_client import ClientRunner
    print("[PASS] ClientRunner class imported successfully.")
except ImportError as e:
    print(f"[FAIL] Could not import hailo_sdk_client: {e}")
    print("If Python version mismatch occurred, verify the wheel is built for your current Python version.")


## 5. Inspect & Verify ONNX Model Asset
Validates the input ONNX model `pcb_multidomain_yolov8n_640.onnx`:
- Verifies SHA-256 checksum (`0f219cae19...`) against the trained baseline
- Confirms input shape `images` $[1, 3, 640, 640]$ Float32
- Confirms output shape `output0` $[1, 10, 8400]$ Float32
- Audits the 6 defect class names stored in model metadata


In [ ]:
import hashlib
import onnx
import onnxruntime as ort
import numpy as np
import shutil
from pathlib import Path

EXPECTED_ONNX_NAME = "pcb_multidomain_yolov8n_640.onnx"
EXPECTED_SHA256 = "0f219cae19807720f6528d635fc7168e1c067a91370e59d165a79c7536089662"

# Locate model in Drive or local workspace
src_onnx = GDRIVE_DIR / EXPECTED_ONNX_NAME
if not src_onnx.exists():
    # Check alternate filenames in Drive
    candidates = list(GDRIVE_DIR.glob("*.onnx"))
    if candidates:
        src_onnx = candidates[0]
        print(f"[NOTE] Using candidate ONNX: {src_onnx.name}")
    else:
        raise FileNotFoundError(f"Please upload '{EXPECTED_ONNX_NAME}' to {GDRIVE_DIR}")

local_onnx = LOCAL_WORK_DIR / EXPECTED_ONNX_NAME
shutil.copy2(src_onnx, local_onnx)
print(f"[+] Staged ONNX locally: {local_onnx}")

# Calculate SHA256
def calc_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

actual_sha = calc_sha256(local_onnx)
print(f"[*] Verified SHA-256: {actual_sha}")
if actual_sha.lower() == EXPECTED_SHA256.lower():
    print("[PASS] SHA-256 matches frozen baseline model.")
else:
    print(f"[WARNING] SHA-256 differs from frozen baseline. Checkpoint variant detected.")

# Audit with ONNX & ONNX Runtime
model = onnx.load(str(local_onnx))
onnx.checker.check_model(model)
print(f"[PASS] ONNX Graph checker passed cleanly (IR v{model.ir_version}, OpSet {[op.version for op in model.opset_import]}).")

session = ort.InferenceSession(str(local_onnx), providers=["CPUExecutionProvider"])
inputs = session.get_inputs()
outputs = session.get_outputs()

print("
--- Model I/O Specification ---")
for inp in inputs:
    print(f"Input : '{inp.name}' | Shape: {inp.shape} | Type: {inp.type}")
for out in outputs:
    print(f"Output: '{out.name}' | Shape: {out.shape} | Type: {out.type}")

# Verify dummy inference
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
res = session.run(None, {inputs[0].name: dummy})
print(f"[PASS] ONNX Runtime execution verified. Output tensor: {res[0].shape}")


## 6. Prepare INT8 Calibration Dataset
Accurate Post-Training Quantization (PTQ) requires authentic calibration frames.  
- **Format:** NumPy array `[N, 640, 640, 3]`, dtype `uint8`, RGB pixel values $[0, 255]$
- **Sample Size:** 256 to 512 images (balances statistical representation with compilation memory)
- **Source:** Loaded from `calib_dataset.npy` in Google Drive, or assembled on-the-fly from a directory of calibration images.


In [ ]:
import cv2
import numpy as np
from pathlib import Path
import shutil

CALIB_FILE_NAME = "calib_dataset.npy"
src_calib = GDRIVE_DIR / CALIB_FILE_NAME
local_calib = LOCAL_WORK_DIR / CALIB_FILE_NAME

if src_calib.exists():
    print(f"[*] Loading calibration dataset from Drive: {src_calib.name}...")
    shutil.copy2(src_calib, local_calib)
    calib_data = np.load(str(local_calib))
else:
    print(f"[*] '{CALIB_FILE_NAME}' not found in Drive. Checking for image folder...")
    # Check if a zip or folder of images is provided
    img_folders = [d for d in GDRIVE_DIR.glob("*") if d.is_dir() or d.suffix == '.zip']
    
    # Generate synthetic/letterbox calibration set if images are placed in Drive
    img_paths = list(GDRIVE_DIR.glob("**/*.jpg")) + list(GDRIVE_DIR.glob("**/*.png"))
    if len(img_paths) >= 16:
        print(f"[+] Found {len(img_paths)} calibration images. Assembling array...")
        selected_paths = img_paths[:512]
        calib_list = []
        for p in selected_paths:
            img = cv2.imread(str(p))
            if img is not None:
                # Resize/letterbox to 640x640 RGB
                h, w = img.shape[:2]
                r = min(640 / w, 640 / h)
                nw, nh = int(round(w * r)), int(round(h * r))
                resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
                canvas = np.full((640, 640, 3), 114, dtype=np.uint8)
                dw, dh = (640 - nw) // 2, (640 - nh) // 2
                canvas[dh:dh+nh, dw:dw+nw] = resized
                # Convert BGR to RGB
                canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
                calib_list.append(canvas_rgb)
        calib_data = np.array(calib_list, dtype=np.uint8)
    else:
        print("[WARNING] No calibration images found in Drive. Generating synthetic calibration set for dry-run testing.")
        print("[CRITICAL] For production deployment, replace with real PCB images to preserve defect detection accuracy!")
        calib_data = np.random.randint(0, 256, size=(128, 640, 640, 3), dtype=np.uint8)

np.save(str(local_calib), calib_data)
# Save copy to Drive for persistence
if not (GDRIVE_DIR / CALIB_FILE_NAME).exists():
    shutil.copy2(local_calib, GDRIVE_DIR / CALIB_FILE_NAME)

print(f"
[PASS] Calibration Dataset Ready:")
print(f"       Shape : {calib_data.shape} [Count, Height, Width, Channels]")
print(f"       Dtype : {calib_data.dtype} (Required: uint8)")
print(f"       Range : Min={calib_data.min()}, Max={calib_data.max()}")
print(f"       Memory: {calib_data.nbytes / (1024*1024):.2f} MB")


## 7. Generate Model Allocation Script (`pcb_yolov8n.alls`)
The `.alls` script instructs Hailo DFC on:
- **Input Normalization:** Automatically divides incoming $[0, 255]$ `uint8` image pixels by $255.0$ in hardware on the NPU input unit.
- **Target Performance Parameters:** Optimizes allocation for 30–60 FPS throughput over PCIe Gen2/3.


In [ ]:
alls_path = LOCAL_WORK_DIR / "pcb_yolov8n.alls"

alls_content = """# EdgePCB-QC YOLOv8n Hailo Model Script
# Bharat AI-SoC Challenge 2026-27 | PS5

# Normalization: YOLOv8 expects pixels scaled from [0, 255] to [0.0, 1.0]
# mean=[0.0, 0.0, 0.0], std=[255.0, 255.0, 255.0]
normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])

# Hardware performance allocation target
performance_param(fps=30)
"""

with open(alls_path, "w") as f:
    f.write(alls_content)

print(f"[+] Created Hailo Model Allocation Script at: {alls_path}")
print("-" * 50)
print(alls_content)
print("-" * 50)

# Save copy to Drive
shutil.copy2(alls_path, GDRIVE_DIR / "pcb_yolov8n.alls")


## 8. Stage 1: Parse ONNX to Hailo Archive (HAR)
Translates the ONNX computational graph into a Hailo Archive representation.

### Strategy Implementation:
- **Strategy A (End-to-End):** Parse full graph targeting `output0` $[1, 10, 8400]$.
- **Strategy B (Raw Detection Heads):** If DFL operations fail during parsing or quantization, parse at the 6 convolution outputs:
  - Stride 8: `/model.22/cv2.0/cv2.0.2/Conv` (64) & `/model.22/cv3.0/cv3.0.2/Conv` (6)
  - Stride 16: `/model.22/cv2.1/cv2.1.2/Conv` (64) & `/model.22/cv3.1/cv3.1.2/Conv` (6)
  - Stride 32: `/model.22/cv2.2/cv2.2.2/Conv` (64) & `/model.22/cv3.2/cv3.2.2/Conv` (6)


In [ ]:
har_path = LOCAL_WORK_DIR / "pcb_multidomain_yolov8n_640.har"

print("=" * 70)
print("STAGE 1: HAILO ONNX PARSER")
print("=" * 70)

# Attempt Strategy A: End-to-End parsing
print("[*] Attempting Strategy A: End-to-End parsing (output0)...")
parse_cmd_a = [
    "hailo", "parser", "onnx", str(local_onnx),
    "--model-name", "pcb_multidomain_yolov8n",
    "--har-path", str(har_path)
]

ret_a = subprocess.run(parse_cmd_a, capture_output=True, text=True)
print(ret_a.stdout)

if ret_a.returncode == 0 and har_path.exists():
    print(f"[PASS] Strategy A succeeded! Generated HAR: {har_path.name}")
else:
    print(f"[WARNING] Strategy A exited with code {ret_a.returncode}: {ret_a.stderr}")
    print("
[*] Falling back to Strategy B: Parsing 6 Raw Detection Heads...")
    
    end_nodes = [
        "/model.22/cv2.0/cv2.0.2/Conv",
        "/model.22/cv3.0/cv3.0.2/Conv",
        "/model.22/cv2.1/cv2.1.2/Conv",
        "/model.22/cv3.1/cv3.1.2/Conv",
        "/model.22/cv2.2/cv2.2.2/Conv",
        "/model.22/cv3.2/cv3.2.2/Conv"
    ]
    
    parse_cmd_b = [
        "hailo", "parser", "onnx", str(local_onnx),
        "--model-name", "pcb_multidomain_yolov8n",
        "--har-path", str(har_path),
        "--start-node-names", "images",
        "--end-node-names", *end_nodes
    ]
    
    ret_b = subprocess.run(parse_cmd_b, capture_output=True, text=True)
    print(ret_b.stdout)
    if ret_b.returncode != 0:
        raise RuntimeError(f"[CRITICAL FAIL] ONNX Parsing failed in both strategies: {ret_b.stderr}")
    print(f"[PASS] Strategy B succeeded! Generated HAR with 6 raw heads: {har_path.name}")

# Persist HAR to Drive
shutil.copy2(har_path, GDRIVE_DIR / har_path.name)
print(f"[+] Saved HAR to Google Drive ({har_path.stat().st_size / (1024*1024):.2f} MB).")


## 9. Stage 2: Post-Training Quantization (INT8 Optimization)
Applies Hailo's hardware-aware quantization algorithm using the calibration dataset.  
- Scales FP32 weights and activations to INT8 precision.
- Validates that dynamic range is preserved without clipping critical fine-trace features.


In [ ]:
quant_har_path = LOCAL_WORK_DIR / "pcb_multidomain_yolov8n_quantized.har"

print("=" * 70)
print("STAGE 2: INT8 POST-TRAINING QUANTIZATION (OPTIMIZATION)")
print("=" * 70)

opt_cmd = [
    "hailo", "optimize", str(har_path),
    "--calib-set-path", str(local_calib),
    "--model-script", str(alls_path),
    "--output-har-path", str(quant_har_path)
]

print(f"[*] Executing command:
    {' '.join(opt_cmd)}
")
ret_opt = subprocess.run(opt_cmd, capture_output=True, text=True)
print(ret_opt.stdout)

if ret_opt.returncode != 0:
    print(f"[ERROR] Optimization failed:
{ret_opt.stderr}")
    raise RuntimeError("INT8 Optimization failed! Review calibration data and model script.")

print(f"[PASS] Optimization and Quantization complete: {quant_har_path.name}")
shutil.copy2(quant_har_path, GDRIVE_DIR / quant_har_path.name)
print(f"[+] Saved Quantized HAR to Google Drive ({quant_har_path.stat().st_size / (1024*1024):.2f} MB).")


## 10. Stage 3: Compile to Target HEF
Compiles the quantized network into physical resource allocation binaries (`.hef`):
1. **Target `hailo8`:** Raspberry Pi AI HAT+ with full Hailo-8 (26 TOPS)
2. **Target `hailo8l`:** Raspberry Pi AI HAT+ with Hailo-8L (13 TOPS)


In [ ]:
out_hef_dir = LOCAL_WORK_DIR / "hef_outputs"
out_hef_dir.mkdir(parents=True, exist_ok=True)

targets = ["hailo8", "hailo8l"]

print("=" * 70)
print("STAGE 3: HAILO HARDWARE COMPILER")
print("=" * 70)

compiled_hefs = []

for tgt in targets:
    print(f"
[*] Compiling for target architecture: {tgt.upper()}...")
    comp_cmd = [
        "hailo", "compiler", str(quant_har_path),
        "--hw-arch", tgt,
        "--output-dir", str(out_hef_dir)
    ]
    
    ret_comp = subprocess.run(comp_cmd, capture_output=True, text=True)
    print(ret_comp.stdout)
    
    if ret_comp.returncode == 0:
        # Find generated HEF
        gen_hefs = list(out_hef_dir.glob(f"*{tgt}*.hef")) or list(out_hef_dir.glob("*.hef"))
        print(f"[PASS] Compilation for {tgt.upper()} succeeded!")
        for h in gen_hefs:
            print(f"       Generated: {h.name} ({h.stat().st_size / (1024*1024):.2f} MB)")
            compiled_hefs.append(h)
    else:
        print(f"[WARNING] Compilation for {tgt} exited with code {ret_comp.returncode}: {ret_comp.stderr}")

print(f"
[+] Total compiled HEF binaries: {len(compiled_hefs)}")


## 11. HEF Inspection & Architecture Validation
Parses the compiled HEF files to inspect:
- Target architecture (`HAILO8` / `HAILO8L`)
- Input virtual stream name and shape
- Output virtual stream names and shapes
- Model name and compilation timestamp


In [ ]:
import subprocess

print("=" * 70)
print("HEF METADATA & STREAM VALIDATION")
print("=" * 70)

all_hefs = list(out_hef_dir.glob("*.hef"))

for hef in all_hefs:
    print(f"
--- Inspecting {hef.name} ---")
    ret = subprocess.run(["hailortcli", "parse-hef", str(hef)], capture_output=True, text=True)
    if ret.returncode == 0:
        print(ret.stdout)
    else:
        # Fallback if hailortcli is not on host (DFC provides hailo visualizer/inspector)
        print(f"File Size: {hef.stat().st_size / (1024*1024):.2f} MB")
        print(f"SHA-256  : {calc_sha256(hef)}")


## 12. Archival & Edge Deployment Package
Copies all compiled HEF binaries and validation logs to Google Drive for immediate download and deployment on the Raspberry Pi 5.


In [ ]:
import shutil

print("=" * 70)
print("EXPORTING ARTIFACTS TO GOOGLE DRIVE")
print("=" * 70)

for hef in out_hef_dir.glob("*.hef"):
    dest = GDRIVE_DIR / hef.name
    shutil.copy2(hef, dest)
    print(f"[+] Saved to Google Drive: {dest.name} ({dest.stat().st_size / (1024*1024):.2f} MB)")

print("
" + "=" * 70)
print("CONVERSION COMPLETE! NEXT STEPS FOR RASPBERRY PI 5 DEPLOYMENT:")
print("=" * 70)
print("1. Download 'pcb_multidomain_yolov8n_640.hef' from Google Drive.")
print("2. Transfer it to your Raspberry Pi 5 at:")
print("   ~/EdgePCB-QC/models/hailo/pcb_multidomain_yolov8n_640.hef")
print("3. Restart the EdgePCB-QC service:")
print("   sudo systemctl restart edgepcb.service")
print("4. Verify hardware inference with:")
print("   python hardware/deployment/verify_hailo.py")
print("=" * 70)
